In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

SEED = 42
TEST_SIZE = 0.2
DATA = "../data/train.csv"
DROP = ["PassengerId", "Name", "Ticket", "Cabin", "SibSp"]

df = pd.read_csv(DATA).drop(columns=DROP)
X, y = df.drop(columns="Survived"), df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

In [12]:
num_cols = ["Age", "Fare", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]


preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh",  OneHotEncoder(handle_unknown="ignore", drop="first")),
    ]), cat_cols),
])

model = Pipeline([
    ("prep", preprocess),
    ("knn",  KNeighborsClassifier(n_neighbors=9)),
])
model.fit(X_train, y_train)


y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]


print(classification_report(y_test, y_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0      0.777     0.855     0.814       110
           1      0.724     0.609     0.661        69

    accuracy                          0.760       179
   macro avg      0.750     0.732     0.738       179
weighted avg      0.757     0.760     0.755       179

ROC-AUC: 0.829
[[94 16]
 [27 42]]


In [ ]:
import sys, os
sys.path.append(os.path.abspath("../"))
from titanic.utils import log_result

log_result(
    model_name="KNN N:9",
    y_true=y_test,
    y_proba=y_proba,      
    y_pred=y_pred,       
    params={"N": "9", "NumericTransformer": "median", "CategoricalTransformer": "most_frequent + OneHotEncoder"},
    notes=""
)

✓ KNN N:9: AUC=0.829, acc=0.7598 → ../results.csv


{'model': 'KNN N:9',
 'timestamp': '2026-11-04 01:34',
 'auc': 0.829,
 'accuracy': 0.7598,
 'f1': 0.6614,
 'precision': 0.7241,
 'recall': 0.6087,
 'params': "{'N': '9', 'NumericTransformer': 'median', 'CategoricalTransformer': 'most_frequent + OneHotEncoder'}",
 'notes': ''}